In [ ]:
from datasets import load_dataset

dataset = load_dataset("civil_comments")

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 1804874
    })
    validation: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 97320
    })
    test: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 97320
    })
})


In [ ]:
train_dataset = dataset["train"]

import pandas as pd

df = train_dataset.to_pandas()

In [ ]:
df['label'] = (df['toxicity'] > 0.5).astype(int) # Create a binary 'label' based on 'toxicity' score

toxic_df = df[df["label"] == 1]
normal_df = df[df["label"] == 0]

toxic_sample = toxic_df.sample(10458, random_state=42)
normal_sample = normal_df.sample(9785, random_state=42)

balanced_df = pd.concat([
    toxic_sample,
    normal_sample
])

In [ ]:
balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [ ]:
print(balanced_df["label"].value_counts())

label
1    10458
0     9785
Name: count, dtype: int64


In [ ]:
balanced_df = balanced_df[["text", "label"]]

In [ ]:
balanced_df.to_csv(
    "toxic_dataset_20k.csv",
    index=False
)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv("toxic_dataset_20k.csv")

print(df.head())

                                                text  label
0  Right. And "pussy grabbing"  is also a common ...      1
1  I am not sure about Canadas chances of an equa...      0
2  It was one of my great peeves that I was a lic...      0
3  While Muslims excoriate us about supposed Isla...      1
4  Seriously, Reg Guard? You're going to name nam...      1


In [ ]:
texts = df["text"].astype(str)
labels = df["label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [ ]:
VOCAB_SIZE = 20000
MAX_LEN = 150
OOV_TOKEN = "<OOV>"

In [ ]:
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token=OOV_TOKEN
)

tokenizer.fit_on_texts(X_train)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

In [ ]:
model = Sequential([

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dense(64, activation='relu'),

    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 79s 364ms/step - accuracy: 0.6949 - loss: 0.5703 - val_accuracy: 0.8490 - val_loss: 0.4153
Epoch 2/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 80s 357ms/step - accuracy: 0.8936 - loss: 0.2867 - val_accuracy: 0.8419 - val_loss: 0.4212
Epoch 3/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 82s 359ms/step - accuracy: 0.9416 - loss: 0.1729 - val_accuracy: 0.8364 - val_loss: 0.4151
Epoch 4/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 83s 365ms/step - accuracy: 0.9694 - loss: 0.1018 - val_accuracy: 0.8296 - val_loss: 0.5478
Epoch 5/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 82s 365ms/step - accuracy: 0.9743 - loss: 0.0845 - val_accuracy: 0.8172 - val_loss: 0.5351


In [ ]:
pred_probs = model.predict(X_test_pad)

predictions = (pred_probs >= 0.5).astype(int)

127/127 ━━━━━━━━━━━━━━━━━━━━ 7s 50ms/step


In [ ]:
print(classification_report(
    y_test,
    predictions
))

              precision    recall  f1-score   support

           0       0.86      0.74      0.79      1957
           1       0.78      0.88      0.83      2092

    accuracy                           0.81      4049
   macro avg       0.82      0.81      0.81      4049
weighted avg       0.82      0.81      0.81      4049



In [ ]:
text = ["Delta, Alaska and Air Canada all serve Seattle to Victoria..... none of whom are likely to stop by."]

seq = tokenizer.texts_to_sequences(text)

pad = pad_sequences(
    seq,
    maxlen=MAX_LEN,
    padding='post'
)

prediction = model.predict(pad)
if prediction[0][0] >= 0.5:
    print(f"{float(prediction[0][0])}% toxique")
else:
    print(f"{float(1-prediction[0][0])}% non toxique")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
0.9916539788246155% non toxique


In [ ]:
def translate(phrase):
  from deep_translator import GoogleTranslator
  traduction = GoogleTranslator(
      source='fr',
      target='en'
  ).translate(phrase)
  return traduction

In [ ]:
def predict(phrase):
  text = translate(phrase)
  seq = tokenizer.texts_to_sequences(text)
  pad = pad_sequences(
    seq,
    maxlen=MAX_LEN,
    padding='post'
  )

  prediction = model.predict(pad)
  if (prediction[0][0] >= 0.5):
    print(f"{float(prediction[0][0])}% toxique")
  else:
    print(f"{float(1-prediction[0][0])}% non toxique")

In [ ]:
phrase= "Delta, Alaska et Air Canada desservent toutes Seattle à Victoria... et aucun d'entre eux n'est susceptible de s'arrêter."
predict(phrase)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
0.9837092161178589% non toxique


### Exporting the Model and Tokenizer

To reuse the trained model and tokenizer, we need to save them. The Keras model can be saved directly using `model.save()`, and the `Tokenizer` object can be saved using Python's `pickle` module.

In [ ]:
# Save the Keras model
model.save('toxicity_model.keras')
print("Model saved as 'toxicity_model.keras'")

Model saved as 'toxicity_model.keras'


In [ ]:
import pickle

# Save the Tokenizer
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Tokenizer saved as 'tokenizer.pkl'")

Tokenizer saved as 'tokenizer.pkl'


In [ ]:
!pip install -U tf2onnx

In [ ]:
import tf2onnx
import onnx
import tensorflow as tf

# Charger le modèle pour s'assurer qu'il est en mémoire
model = tf.keras.models.load_model('toxicity_model.keras')

# Définir la signature d'entrée
# Votre MAX_LEN est 150
input_signature = [tf.TensorSpec([None, 150], tf.float32, name='input')]

# Conversion du modèle Keras vers ONNX
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature, opset=13)

# Sauvegarder le fichier .onnx
onnx.save(onnx_model, "toxicity_model.onnx")

print("Modèle converti avec succès en 'toxicity_model.onnx'")

In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.7 MB/s eta 0:00:00


In [ ]:
from deep_translator import GoogleTranslator
phrase = "Delta, Alaska et Air Canada desservent toutes Seattle à Victoria... et aucun d'entre eux n'est susceptible de s'arrêter."
traduction = GoogleTranslator(
    source='fr',
    target='en'
).translate(phrase)
print(traduction)

Delta, Alaska, and Air Canada all fly from Seattle to Victoria...and none of them are likely to stop.
